## 0 · Configuration & Imports

In [0]:
import logging
import json
from datetime import datetime, timezone
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, IntegerType, TimestampType, ArrayType
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("bronze_layer")

spark = SparkSession.builder.appName("youtube_bronze").getOrCreate()



## 1 · Pipeline Parameters

In [0]:
dbutils.widgets.text("bucket_name",  "youtube-capstone-bucket-team6", "S3 Bucket Name")
dbutils.widgets.text("catalog_name", "youtube-capstone-catalog",     "Unity Catalog Name")
dbutils.widgets.text("run_date",     datetime.now(timezone.utc).strftime("%Y-%m-%d"), "Run Date")

BUCKET_NAME  = dbutils.widgets.get("bucket_name")
CATALOG_NAME = dbutils.widgets.get("catalog_name")
RUN_DATE     = dbutils.widgets.get("run_date")

BASE_PATH           = f"s3://{BUCKET_NAME}/youtube_pipeline"
RAW_VIDEOS_PATH     = f"{BASE_PATH}/raw_uploads/raw_videos/run_date={RUN_DATE}"
RAW_COMMENTS_PATH   = f"{BASE_PATH}/raw_uploads/raw_comments/run_date={RUN_DATE}"
BRONZE_PATH         = f"{BASE_PATH}/bronze"
CHECKPOINT_PATH     = f"{BASE_PATH}/checkpoints/bronze"

BRONZE_DB           = f"{CATALOG_NAME}.bronze"

logger.info(f"Bronze config → catalog={CATALOG_NAME}, run_date={RUN_DATE}")
logger.info(f"Bronze output → {BRONZE_PATH}")

## 2 · Schema Definitions

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType


VIDEO_SCHEMA = StructType([
    StructField("video_id",       StringType(),    nullable=False),
    StructField("title",          StringType(),    nullable=True),
    StructField("channel_id",     StringType(),    nullable=True),
    StructField("channel_title",  StringType(),    nullable=True),
    StructField("publish_date",   StringType(),    nullable=True),  
    StructField("description",    StringType(),    nullable=True),
    StructField("tags",           StringType(),    nullable=True),  
    StructField("category_id",    StringType(),    nullable=True),
    StructField("duration",       StringType(),    nullable=True),  # ISO-8601, parsed in Silver
    StructField("thumbnail_url",  StringType(),    nullable=True),
    StructField("view_count",     LongType(),      nullable=True),
    StructField("like_count",     LongType(),      nullable=True),
    StructField("comment_count",  LongType(),      nullable=True),
    StructField("region_code",    StringType(),    nullable=True),
    StructField("run_date",       StringType(),    nullable=True),
    StructField("ingested_at",    StringType(),    nullable=True),
])

COMMENT_SCHEMA = StructType([
    StructField("comment_id",   StringType(), nullable=False),
    StructField("video_id",     StringType(), nullable=False),
    StructField("author",       StringType(), nullable=True),
    StructField("comment_text", StringType(), nullable=True),
    StructField("like_count",   LongType(),   nullable=True),
    StructField("comment_date", StringType(), nullable=True),
    StructField("run_date",     StringType(), nullable=True),
    StructField("ingested_at",  StringType(), nullable=True),
])

## 3 · Helper Functions

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType

def read_raw_parquet(
    path: str,
    schema: StructType,
    checkpoint_path: str,
    table_label: str
) -> DataFrame:
    

    try:
        df = (
            spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", "parquet")
                .option("cloudFiles.schemaLocation", checkpoint_path)
                .schema(schema)
                .load(path)
        )

        logger.info(
            f"[{table_label}] Streaming source initialized from {path}"
        )

        return df

    except Exception as exc:
        logger.error(
            f"[{table_label}] Failed to read from {path}: {exc}"
        )
        raise

def add_bronze_metadata(df: DataFrame, source_label: str) -> DataFrame:
    
    return (
        df
        .withColumn("_bronze_ingested_at", F.current_timestamp())
        .withColumn("_source_layer",       F.lit("raw_uploads"))
        .withColumn("_source_label",       F.lit(source_label))
    )





from pyspark.sql import DataFrame

def write_bronze_delta(
    df: DataFrame,
    path: str,
    checkpoint_path: str,
    catalog_table: str,
    partition_cols: list[str],
    table_label: str,
) -> int:
    

    parts = catalog_table.split(".")
    quoted_table = f"`{parts[0]}`.`{parts[1]}`.`{parts[2]}`"

    (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .partitionBy(*partition_cols)
        .trigger(availableNow=True)      # process backlog once and stop
        .start(path)
        .awaitTermination()
    )

    # Register table
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {quoted_table}
        USING DELTA
        LOCATION '{path}'
    """)

    spark.sql(f"OPTIMIZE {quoted_table}")

    count = spark.read.format("delta").load(path).count()

    logger.info(
        f"[{table_label}] Appended {count:,} rows to {path}"
    )

    return count

## 4 · Bronze Videos

In [0]:
logger.info("── Building bronze_videos ───")

spark.sql(f"CREATE DATABASE IF NOT EXISTS `{CATALOG_NAME}`.bronze")


bronze_videos_path   = f"{BRONZE_PATH}/bronze_videos"
videos_checkpoint    = f"{CHECKPOINT_PATH}/bronze_videos"

df_raw_videos = read_raw_parquet(
    path            = RAW_VIDEOS_PATH,
    schema          = VIDEO_SCHEMA,
    checkpoint_path = videos_checkpoint,
    table_label     = "raw_videos",
)

# 4b. Add audit metadata columns
df_raw_videos = add_bronze_metadata(df_raw_videos, "raw_videos")


# 4d. Light casting (no business logic – that belongs in Silver)
df_bronze_videos = (
    df_raw_videos
    .withColumn("view_count",    F.col("view_count").cast(LongType()))
    .withColumn("like_count",    F.col("like_count").cast(LongType()))
    .withColumn("comment_count", F.col("comment_count").cast(LongType()))
)

# 4e. Write incrementally (append-only) via write_bronze_delta
v_count = write_bronze_delta(
    df             = df_bronze_videos,
    path           = bronze_videos_path,
    checkpoint_path= videos_checkpoint,
    catalog_table  = f"{CATALOG_NAME}.bronze.bronze_videos",
    partition_cols = ["region_code", "run_date"],
    table_label    = "bronze_videos",
)




df_bronze_videos = spark.read.format("delta").load(bronze_videos_path)
df_bronze_videos.printSchema()


root
 |-- video_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_id: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- publish_date: string (nullable = true)
 |-- description: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- thumbnail_url: string (nullable = true)
 |-- view_count: long (nullable = true)
 |-- like_count: long (nullable = true)
 |-- comment_count: long (nullable = true)
 |-- region_code: string (nullable = true)
 |-- run_date: string (nullable = true)
 |-- ingested_at: string (nullable = true)
 |-- _bronze_ingested_at: timestamp (nullable = true)
 |-- _source_layer: string (nullable = true)
 |-- _source_label: string (nullable = true)



## 5 · Bronze Comments

In [0]:
logger.info("── Building bronze_comments ─────")

bronze_comments_path  = f"{BRONZE_PATH}/bronze_comments"
comments_checkpoint   = f"{CHECKPOINT_PATH}/bronze_comments"

df_raw_comments = read_raw_parquet(
    path            = RAW_COMMENTS_PATH,
    schema          = COMMENT_SCHEMA,
    checkpoint_path = comments_checkpoint,
    table_label     = "raw_comments",
)

# 5b. Add audit metadata columns
df_raw_comments = add_bronze_metadata(df_raw_comments, "raw_comments")


df_bronze_comments = df_raw_comments.withColumn("like_count", F.col("like_count").cast(LongType()))

# 5e. Write incrementally (append-only) via write_bronze_delta
c_count = write_bronze_delta(
    df             = df_bronze_comments,
    path           = bronze_comments_path,
    checkpoint_path= comments_checkpoint,
    catalog_table  = f"{CATALOG_NAME}.bronze.bronze_comments",
    partition_cols = ["run_date"],
    table_label    = "bronze_comments",
)




df_bronze_comments = spark.read.format("delta").load(bronze_comments_path)
df_bronze_comments.printSchema()


root
 |-- comment_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- comment_text: string (nullable = true)
 |-- like_count: long (nullable = true)
 |-- comment_date: string (nullable = true)
 |-- run_date: string (nullable = true)
 |-- ingested_at: string (nullable = true)
 |-- _bronze_ingested_at: timestamp (nullable = true)
 |-- _source_layer: string (nullable = true)
 |-- _source_label: string (nullable = true)



## 6 · Data Quality Checks

In [0]:
def run_dq_checks(df: DataFrame, table_name: str, not_null_cols: list[str]) -> None:
    """
    Lightweight inline DQ checks.
    Logs a WARNING for any violation — does NOT raise by default
    (change to raise for strict mode).
    """
    total = df.count()
    logger.info(f"[DQ] {table_name}: {total:,} rows")

    for col_name in not_null_cols:
        null_count = df.filter(F.col(col_name).isNull()).count()
        if null_count > 0:
            logger.warning(f"[DQ] {table_name}.{col_name}: {null_count:,} NULL values ({null_count/total:.1%})")
        else:
            logger.info(f"[DQ] {table_name}.{col_name}: ✓ no nulls")

bv = spark.read.format("delta").load(f"{BRONZE_PATH}/bronze_videos")
run_dq_checks(bv, "bronze_videos",   ["video_id", "channel_id", "view_count"])

bc = spark.read.format("delta").load(f"{BRONZE_PATH}/bronze_comments")
run_dq_checks(bc, "bronze_comments", ["comment_id", "video_id"])

## 7 · Summary

In [0]:
print("=" * 60)
print("  BRONZE LAYER COMPLETE — SUMMARY")
print("=" * 60)
print(f"  Run Date            : {RUN_DATE}")
print(f"  bronze_videos rows  : {v_count:,}")
print(f"  bronze_comments rows: {c_count:,}")
print(f"  Catalog             : {BRONZE_DB}")
print(f"  Delta location      : {BRONZE_PATH}")
print("=" * 60)

dbutils.notebook.exit(json.dumps({
    "status":                "success",
    "run_date":              RUN_DATE,
    "bronze_videos_count":   v_count,
    "bronze_comments_count": c_count,
}))